# Step 8: full check of the team repo with the study in it
### Neeraj, MMAI 894 (router fine-tuning)

**Why:** the study now lives in the team repo (branch `finetune-study` on my fork). My laptop could not run the
whole team flow (no installs there), so this notebook runs it the way a TA would: fresh copy, install, every stage, every test.

**Settings:** Accelerator **None** (CPU is enough), Internet **On**, no secrets attached. Nothing here can spend money:
every model call replays from the recordings and the key is removed.

**What it does:** 1) clone the branch, 2) install, 3) run the evaluation, fine-tune and economics stages (replay),
4) run all tests, 5) check that the replayed numbers equal the committed ones, 6) optional: run the fine-tuned model
itself on CPU and check it makes the decisions it recorded.

In [1]:
import os
os.environ.pop("OPENROUTER_API_KEY", None)     # never needed: everything replays from the recordings
os.environ["NORTHSTAR_LLM_MODE"] = "replay"
!rm -rf /kaggle/working/northstar-agent
!git clone -q --branch finetune-study https://github.com/neerajmehla28/northstar-agent.git /kaggle/working/northstar-agent
%cd /kaggle/working/northstar-agent
!git log --oneline -4

/kaggle/working/northstar-agent
68549a7 (HEAD -> finetune-study, origin/finetune-study) Report pages 4, 5 and 7, docs and decision log for the fine-tuning study
8f81092 Add the study's two routers to the final evaluation and the cost model (D-36)
7d880e2 Add the fine-tuning study: code, frozen settings, results and notebooks
0cc6208 (origin/main, origin/HEAD) Merge pull request #51 from Salikawtar/chore/clean-internal-notes


In [2]:
!pip install -q -r requirements.txt 2>&1 | tail -3
!python -c "import faker, pydantic, scipy, sklearn, pandas; print('installed: faker', faker.VERSION, '| pydantic', pydantic.VERSION)"

installed: faker 40.63.0 | pydantic 2.14.1


### 3. The team flow, replayed (about 5 minutes)
`evaluate`: validation, test (10 setups), fresh, grader checks. `finetune`: the three-router comparison and the study's tables.
`economics`: the cost model. The last lines should show `new_spend_usd: 0.0`.

In [3]:
!python run_all.py evaluate finetune economics > /kaggle/working/run_all.log 2>&1; echo "exit code: $?"
!grep -n "Model calls\|Traceback\|Error" /kaggle/working/run_all.log | head -20
!tail -5 /kaggle/working/run_all.log

exit code: 0
12:Model calls: {'total': 186, 'replayed': 186, 'new_spend_usd': 0.0}
49:Model calls: {'total': 972, 'replayed': 972, 'new_spend_usd': 0.0}
68:Model calls: {'total': 249, 'replayed': 249, 'new_spend_usd': 0.0}
--- python -m northstar.finetune.report_figure

=== economics ===

--- python -m northstar.economics


### 4. Every test (about 3 minutes)

In [4]:
!python -m pytest -q -p no:cacheprovider -rf 2>&1 | tail -15

tests/test_economics.py::test_breakeven_volume_is_where_the_saving_changes_sign
tests/test_economics.py::test_breakeven_volume_is_where_the_saving_changes_sign
tests/test_economics.py::test_breakeven_volume_is_where_the_saving_changes_sign
tests/test_economics.py::test_breakeven_volume_is_where_the_saving_changes_sign
  /usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
    parser.resetCache()

tests/test_economics.py::test_breakeven_volume_is_where_the_saving_changes_sign
  /usr/local/lib/python3.13/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
    ParserElement.enablePackrat()

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
=========================== short test summary info ============================
FAILED tests/test_finetune.py::test_lora_trains_far_fewer_weights_and_never_touches_t

### 5. Did the replay give the committed numbers?
Every setup's task success, rebuilt here, against the file in the branch. `same` should be True everywhere.
Only latencies may move a little (replay adds local compute time).

In [5]:
import json, subprocess
import pandas as pd

def committed(path):
    return json.loads(subprocess.run(["git", "show", f"HEAD:{path}"], capture_output=True, text=True).stdout)

rows = []
for split, name in [("test", "summary.json"), ("fresh", "summary_fresh.json")]:
    path = f"src/northstar/evaluation/results/{name}"
    now, then = json.load(open(path)), committed(path)
    print(split, "model calls:", now["model_calls"])
    for k, v in now["setups"].items():
        before = then["setups"].get(k, {}).get("task_success", {}).get("rate")
        rows.append(dict(split=split, setup=k, task_success=v["task_success"]["rate"], committed=before,
                         same=v["task_success"]["rate"] == before))
checks = pd.DataFrame(rows)
print("all the same:", bool(checks.same.all()))
checks

test model calls: {'total': 972, 'replayed': 972, 'new_spend_usd': 0.0}
fresh model calls: {'total': 249, 'replayed': 249, 'new_spend_usd': 0.0}
all the same: True


,split,setup,task_success,committed,same
0,test,full_agent,0.700,0.700,True
1,test,full_agent_before_fixes,0.525,0.525,True
2,test,full_agent_router_sonnet,0.650,0.650,True
3,test,full_agent_router_finetuned,0.400,0.400,True
4,test,full_agent_router_finetuned_lora,0.575,0.575,True
5,test,full_agent_router_jev,0.550,0.550,True
6,test,full_agent_router_v2,0.675,0.675,True
7,test,full_agent_warranty_high,0.650,0.650,True
8,test,full_agent_no_guardrails,0.700,0.700,True
9,test,single_agent,0.450,0.450,True


In [6]:
!git status --short | grep -v "^??" | head -40

 M src/northstar/economics/results/cost_per_chat_vs_volume.png
 M src/northstar/economics/results/cost_per_chat_vs_volume_page7.png
 M src/northstar/economics/results/sensitivity.csv
 M src/northstar/economics/results/tables.md
 M src/northstar/evaluation/results/economics_inputs.json
 M src/northstar/evaluation/results/headline.csv
 M src/northstar/evaluation/results/headline.md
 M src/northstar/evaluation/results/headline_fresh.csv
 M src/northstar/evaluation/results/headline_fresh.md
 M src/northstar/evaluation/results/per_chat.csv
 M src/northstar/evaluation/results/per_chat_fresh.csv
 M src/northstar/evaluation/results/per_chat_validation.csv
 M src/northstar/evaluation/results/summary.json
 M src/northstar/evaluation/results/summary_fresh.json
 M src/northstar/evaluation/results/summary_validation.json
 M src/northstar/finetune/results/breakeven.json
 M src/northstar/finetune/results/datasize_curve.png
 M src/northstar/finetune/results/datasize_summary.json
 M src/northstar/finet

### 6. Optional: the fine-tuned model itself, on CPU (about 3 minutes, downloads xlm-roberta-base, 1 GB)
The evaluation replays the router's recorded decisions. This runs the trained model (five LoRA seeds, 6 MB) on the
40 test chats and checks it makes the same decisions. It opens the test chats (logged in the study's access log),
but scores nothing and chooses nothing.

In [7]:
!pip install -q -r requirements-finetune.txt 2>&1 | tail -2
!pip uninstall -y -q torchao     # Kaggle's torchao breaks peft's import (seen in step 3)

In [8]:
%%writefile /kaggle/working/live_check.py
import os, sys
import pandas as pd
os.environ["NORTHSTAR_ALLOW_TEST"] = "1"
sys.path.insert(0, "src")
from northstar.finetune.study import protocol as P
from northstar.finetune.study.handover_router import TunedRouter
from northstar.evaluation.setups import STUDY_RECORDINGS

rec = pd.read_csv(STUDY_RECORDINGS / "router_finetuned-balanced_test_and_fresh.csv").set_index("id")
test = P.test_set(P.load_all(), final=True, note="notebook 08: the live model against its own recorded decisions (no scoring)")
router = TunedRouter()
rows = []
for cid, text in zip(test["id"], P.input_text(test, "first_message")):
    p, r = router.predict(text), rec.loc[cid]
    rows.append(dict(id=cid, live_type=p["request_type"], recorded_type=r.pred_type,
                     same_type=p["request_type"] == r.pred_type,
                     same_person=p["needs_escalation"] == (str(r.pred_escalation).lower() == "true"),
                     same_reason=p["escalation_reason"] == r.pred_reason,
                     confidence_gap=abs(p["confidence"] - float(r.confidence)), ms=p["latency_ms"]))
d = pd.DataFrame(rows)
print(f"{len(d)} test chats: request type same {d.same_type.mean():.1%}, person needed same {d.same_person.mean():.1%}, "
      f"reason same {d.same_reason.mean():.1%}")
print(f"largest confidence gap {d.confidence_gap.max():.4f}; median {d.ms.median():.0f} ms a chat (5 seeds, CPU)")
diff = d[~(d.same_type & d.same_person & d.same_reason)]
print("chats that differ:", len(diff))
print(diff.to_string() if len(diff) else "none")

Writing /kaggle/working/live_check.py


In [9]:
!python /kaggle/working/live_check.py
!PYTHONPATH=src python -m northstar.finetune.study.handover_router "My car won't start and I'm stuck on the highway near exit 40."


model.safetensors: downloading bytes: ██▎                   |  120MB, 8.89MB/s  
model.safetensors: downloading bytes: ██▊                   |  142MB, 11.2MB/s  
model.safetensors: downloading bytes: ████████▎             |  421MB, 34.0MB/s  
model.safetensors: downloading bytes: ██████████▏           |  516MB, 41.3MB/s  
model.safetensors: downloading bytes: ██████████▉           |  552MB, 45.0MB/s  
model.safetensors: downloading bytes: ██████████████▏       |  722MB, 56.6MB/s  
model.safetensors: reconstructing file:  84%|█████ |  938MB / 1.12GB, 67.0MB/s  
model.safetensors: reconstructing file:  92%|█████▌| 1.02GB / 1.12GB, 77.2MB/s  
model.safetensors: downloading bytes: ██████████████████████|  745MB, 59.2MB/s  
model.safetensors: reconstructing file: 100%|██████| 1.12GB / 1.12GB, 91.4MB/s  
Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 2700.09it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  

**Send back:** the outputs of steps 3 to 6 (a screenshot or the printed text), or download the notebook with its outputs.